# Notebook 03 — Feature Engineering

**Purpose:**  
Enrich the merged bid dataset with derived features and backfill missing values.

**Inputs** (`data/processed/`):
- `merged_bid_data.xlsx`
- `merged_financial_evaluation.xlsx`

**Also reads** (`data/raw/`) for title backfill:
- `coal_india_tenders_details.xlsx`
- `eprocure_result_tenders_detail.xlsx`

**Output** (`data/processed/`):
- `final_bid_data.xlsx` — fully enriched bid table
- `final_FE.xlsx` — financial eval with `drilling_meterage` and `per_meter_rate`

**Features engineered:**

| Feature | Method |
|---------|--------|
| `title_description` (backfill) | Map from raw `Title/Items` column for Coal India and eProcure |
| `mineral_name` | Keyword search — 15-mineral reference list |
| `state_name` (backfill) | Keyword search against full state name list |
| `drilling_meterage` | 3-tier regex: direct qty → BH×depth → generic fallback |
| `work_order` | Contract URL mapped from GEM tech eval file |
| `winner` (imputation) | L1-rank imputation where awarded bidder not published |
| `winning_price` (imputation) | L1-rank imputation |
| `tender_value` (backfill) | Filled from minimum price in financial_eval |
| `per_meter_rate` | `drilling_meterage × price` added to financial_eval |

---

## 1. Imports & Setup

In [1]:
import sys
import re
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from config import RAW_FILES, PROCESSED_FILES, DATA_RAW
from src.extractors import extract_mineral, extract_state_from_title
from src.parsers import extract_drilling_meterage
from src.cleaning import clean_company_name

print('Setup complete.')

Setup complete.


## 2. Load Data

In [2]:
bid_data = pd.read_excel(PROCESSED_FILES['merged_bids'])
fe_data  = pd.read_excel(PROCESSED_FILES['merged_fe'])

bid_data['bid_start_date'] = pd.to_datetime(bid_data['bid_start_date'], errors='coerce')
bid_data['bid_end_date']   = pd.to_datetime(bid_data['bid_end_date'],   errors='coerce')
bid_data.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
fe_data.drop( columns=['Unnamed: 0'], inplace=True, errors='ignore')

# Defensive re-check: fe_data['seller'] should already be clean coming out of
# Notebook 02 (which cleans it immediately after the portal merge). We apply
# clean_company_name again here anyway because it is idempotent and this
# column is about to be used by the L1 winner imputation in Section 7, which
# writes directly into bid_data['winner']. If Notebook 02 is ever skipped or
# an older merged_financial_evaluation.xlsx is loaded by mistake, this line
# is what stops uncleaned names from reaching bid_data via that path.
fe_data['seller'] = fe_data['seller'].astype(str).str.strip().apply(clean_company_name)

print(f"bid_data : {bid_data.shape}")
print(f"fe_data  : {fe_data.shape}")

bid_data : (5689, 13)
fe_data  : (22519, 5)


In [3]:
# Guard: every downstream .map() / .loc[mask] in this notebook keys on bid_number.
# If bid_number is not unique, those operations can silently misalign winner,
# title_description, and state_name across rows. Fail loudly here instead.
assert bid_data['bid_number'].is_unique, (
    "bid_number is not unique in merged_bid_data.xlsx -- re-run Notebook 02, "
    "which deduplicates on bid_number before export."
)
print("Guard passed: bid_number is unique.")

Guard passed: bid_number is unique.


## 3. Backfill Missing Titles

Some Coal India and eProcure bids arrived without a title in the standardised extract.
We look up `Title/Items` directly from the raw source files.

In [4]:
missing_title = bid_data[bid_data['title_description'].isna()]
print(f"Missing titles before backfill: {len(missing_title)}")

# Coal India backfill
coal_raw  = pd.read_excel(RAW_FILES['coal_india'])
title_map = (
    coal_raw[coal_raw['Bid Number'].isin(missing_title['bid_number'])]
    .drop_duplicates(subset='Bid Number')
    .set_index('Bid Number')['Title/Items']
)
bid_data['title_description'] = bid_data['title_description'].fillna(
    bid_data['bid_number'].map(title_map)
)

# eProcure backfill
eprocure_raw = pd.read_excel(RAW_FILES['eprocure'])
title_map    = (
    eprocure_raw
    .drop_duplicates(subset='Bid Number')
    .set_index('Bid Number')['Title/Items']
)
bid_data['title_description'] = bid_data['title_description'].fillna(
    bid_data['bid_number'].map(title_map)
)

print(f"Missing titles after  backfill: {bid_data['title_description'].isna().sum()}")

Missing titles before backfill: 6
Missing titles after  backfill: 0


## 4. Mineral Name Extraction

Searches each tender title for keywords from a 15-mineral reference list.
Returns the first match, or NaN. See `src/extractors.py`.

In [5]:
bid_data['mineral_name'] = bid_data['title_description'].apply(extract_mineral)

n_found = bid_data['mineral_name'].notna().sum()
print(f"Tenders with identified mineral: {n_found} / {len(bid_data)} ({n_found/len(bid_data)*100:.1f}%)")
print("\nMineral distribution:")
print(bid_data['mineral_name'].value_counts().to_string())

Tenders with identified mineral: 1404 / 5689 (24.7%)

Mineral distribution:
mineral_name
Coal         1053
Manganese     131
Copper         88
Iron           63
Bauxite        50
Gypsum         13
Limestone       3
Gold            1
Dolomite        1
Silica          1


## 5. State Name Backfill

State was inferred from org hierarchy for eProcure in Notebook 01.
For remaining blanks, keyword search the tender title.

In [6]:
n_before = bid_data['state_name'].isna().sum()

mask = bid_data['state_name'].isna()
bid_data.loc[mask, 'state_name'] = (
    bid_data.loc[mask, 'title_description'].apply(extract_state_from_title)
)

n_after = bid_data['state_name'].isna().sum()
print(f"state_name null before backfill : {n_before}")
print(f"state_name null after  backfill : {n_after}")
print(f"States resolved from title      : {n_before - n_after}")

state_name null before backfill : 5379
state_name null after  backfill : 5132
States resolved from title      : 247


## 6. Drilling Meterage Extraction

Extracts total drilling quantity (metres) from tender titles using a 3-tier
priority cascade. See `src/parsers.py` for full documentation.

In [7]:
bid_data['drilling_meterage'] = bid_data['title_description'].apply(extract_drilling_meterage)

n_found = bid_data['drilling_meterage'].notna().sum()
print(f"Tenders with drilling meterage : {n_found} / {len(bid_data)} ({n_found/len(bid_data)*100:.1f}%)")
valid = bid_data['drilling_meterage'].dropna()
if len(valid) > 0:
    print(f"Meterage range                 : {valid.min():,.0f} m  —  {valid.max():,.0f} m")

Tenders with drilling meterage : 564 / 5689 (9.9%)
Meterage range                 : 0 m  —  353,800 m


## 7. Winner Imputation from L1 Bidder

Some tenders have a known L1 bidder in financial_eval but no Awarded Bidder on the portal.
Where there is exactly one L1 bidder, they are imputed as the winner.

In [8]:
# Only use bid_numbers with a single unambiguous L1 bidder
l1_data = fe_data[fe_data['rank_position'] == 'L1'].copy()

l1_counts     = l1_data.groupby('bid_number')['seller'].nunique()
unambiguous   = l1_counts[l1_counts == 1].index

l1_ref = (
    l1_data[l1_data['bid_number'].isin(unambiguous)]
    .drop_duplicates(subset='bid_number')
    .set_index('bid_number')
)

missing_mask = bid_data['winner'].isna()
n_to_impute  = missing_mask.sum()

bid_data.loc[missing_mask, 'winner']        = bid_data.loc[missing_mask, 'bid_number'].map(l1_ref['seller'])
bid_data.loc[missing_mask, 'winning_price'] = bid_data.loc[missing_mask, 'bid_number'].map(l1_ref['price'])

n_still_missing = bid_data['winner'].isna().sum()
print(f"Tenders missing winner before imputation : {n_to_impute}")
print(f"Winners imputed from L1 bidder           : {n_to_impute - n_still_missing}")
print(f"Tenders still missing winner             : {n_still_missing}")

Tenders missing winner before imputation : 469
Winners imputed from L1 bidder           : 418
Tenders still missing winner             : 51


## 8. Tender Value Backfill

Where `tender_value` is still missing, fill from the minimum price in financial_eval
for that tender (a reasonable proxy for estimated value).

In [9]:
min_price_map = fe_data.groupby('bid_number')['price'].min()

n_missing_tv = bid_data['tender_value'].isna().sum()
mask = bid_data['tender_value'].isna()
bid_data.loc[mask, 'tender_value'] = bid_data.loc[mask, 'bid_number'].map(min_price_map)

resolved = n_missing_tv - bid_data['tender_value'].isna().sum()
print(f"tender_value backfilled from financial_eval : {resolved}")
print(f"tender_value still missing                  : {bid_data['tender_value'].isna().sum()}")

tender_value backfilled from financial_eval : 912
tender_value still missing                  : 8


## 9. GEM Work Order URL

Maps contract URLs from the GEM tech eval file onto bid_data as `work_order`.
If the tech eval file is absent, the column is set to NaN and the pipeline continues.

In [10]:
tech_eval_path = DATA_RAW / 'gem_specific_tech_eval_output.xlsx'

if tech_eval_path.exists():
    tech_eval = pd.read_excel(tech_eval_path)
    bid_url   = tech_eval[['Bid No', 'View Contract URL']].drop_duplicates(subset='Bid No')
    url_map   = bid_url.set_index('Bid No')['View Contract URL']
    bid_data['work_order'] = bid_data['bid_number'].map(url_map)
    print(f"work_order URLs mapped: {bid_data['work_order'].notna().sum()}")
else:
    bid_data['work_order'] = np.nan
    print("gem_specific_tech_eval_output.xlsx not found — work_order set to NaN.")

work_order URLs mapped: 3223


## 10. Add drilling_meterage & per_meter_rate to Financial Eval

Map `drilling_meterage` from bid_data onto financial_eval (keyed on bid_number),
then compute `per_meter_rate = drilling_meterage × price`.

In [11]:
meterage_map = bid_data.set_index('bid_number')['drilling_meterage']

fe_data['drilling_meterage'] = fe_data['bid_number'].map(meterage_map)
fe_data['per_meter_rate']    = fe_data['drilling_meterage'] * fe_data['price']

n_with_rate = fe_data['per_meter_rate'].notna().sum()
print(f"FE rows with per_meter_rate populated: {n_with_rate} / {len(fe_data)}")

FE rows with per_meter_rate populated: 1756 / 22519


## 11. Data Quality Summary

In [12]:
quality = pd.DataFrame({
    'missing_count'   : bid_data.isna().sum(),
    'completeness_pct': (bid_data.notna().mean() * 100).round(1),
    'dtype'           : bid_data.dtypes,
}).sort_values('completeness_pct')

print("Data Quality Report — final bid_data")
print("=" * 50)
print(quality.to_string())
print(f"\nTotal rows: {len(bid_data)}")

Data Quality Report — final bid_data
                   missing_count  completeness_pct           dtype
state_name                  5132               9.8          object
drilling_meterage           5125               9.9         float64
mineral_name                4285              24.7          object
contract_period             2514              55.8         float64
work_order                  2466              56.7          object
award_status                 187              96.7          object
winner                        51              99.1          object
winning_price                 51              99.1         float64
tender_value                   8              99.9         float64
bid_number                     0             100.0          object
title_description              0             100.0          object
ministry                       0             100.0          object
department_org                 0             100.0          object
bid_start_date           

## 12. Export

In [13]:
# Set bid_number as index (matches your original notebook convention)
bid_data_export = bid_data.copy()
bid_data_export.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
bid_data_export.set_index('bid_number', drop=True, inplace=True)
bid_data_export.to_excel(PROCESSED_FILES['featured_bids'])

fe_data.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
fe_data.to_excel(PROCESSED_FILES['merged_fe'], index=False)

print(f"Exported final_bid_data  →  {PROCESSED_FILES['featured_bids'].name}  ({len(bid_data_export)} rows)")
print(f"Exported final_FE        →  {PROCESSED_FILES['merged_fe'].name}  ({len(fe_data)} rows)")
print("\nNotebook 03 complete. Proceed to 04_seller_master.ipynb.")

Exported final_bid_data  →  final_bid_data.xlsx  (5689 rows)
Exported final_FE        →  merged_financial_evaluation.xlsx  (22519 rows)

Notebook 03 complete. Proceed to 04_seller_master.ipynb.
